# Graph Neural Network (GNN) basics

A GNN classifies or predicts on graph-structured data by repeatedly having each node
**aggregate its neighbors' representations** - "message passing" - so a node's final
representation encodes not just its own features but its local neighborhood's structure too.

To make the benefit obvious, we deliberately give each node a *noisy, individually weak*
signal, then show that a GNN can denoise it by averaging over structurally-related neighbors -
something a model with no access to the graph cannot do at all.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

N = 60          # nodes per graph
P_IN = 0.25     # edge probability within the same community
P_OUT = 0.03    # edge probability across communities
NOISE_STD = 1.6 # per-node feature noise - deliberately large


def make_graph(rng):
    """A 2-community stochastic block model graph. Each node's "feature" is its true
    community (+-1) buried in heavy Gaussian noise - individually almost useless."""
    labels = rng.randint(0, 2, size=N)
    adj = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        for j in range(i + 1, N):
            p = P_IN if labels[i] == labels[j] else P_OUT
            if rng.rand() < p:
                adj[i, j] = adj[j, i] = 1.0
    signal = np.where(labels == 1, 1.0, -1.0)
    features = signal + rng.randn(N) * NOISE_STD
    return adj, features.astype(np.float32), labels


def normalize_adj(adj):
    """Mean-aggregation matrix: each row sums to 1 over that node's neighbors."""
    deg = adj.sum(axis=1, keepdims=True)
    deg[deg == 0] = 1.0
    return adj / deg


# Sanity check: how good is a plain threshold on the raw noisy feature alone (no graph)?
rng = np.random.RandomState(0)
accs = []
for _ in range(20):
    _, feat, lab = make_graph(rng)
    pred = (feat > 0).astype(int)
    accs.append((pred == lab).mean())
print(f"raw-feature threshold accuracy (no graph info): {np.mean(accs):.3f} +/- {np.std(accs):.3f}")

## The models

`GCNLayer` combines each node's own (transformed) feature with the mean of its neighbors'
(transformed) features - the simplest form of graph convolution. `MLPBaseline` sees only a
node's own feature and has no access to the graph at all, so any gap between the two is
entirely down to using graph structure.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj_norm):
        agg = adj_norm @ x
        return torch.relu(self.lin_self(x) + self.lin_neigh(agg))


class GCN(nn.Module):
    def __init__(self, hidden=16, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        d = 1
        for _ in range(n_layers):
            self.layers.append(GCNLayer(d, hidden))
            d = hidden
        self.head = nn.Linear(hidden, 2)

    def forward(self, x, adj_norm):
        h = x
        for layer in self.layers:
            h = layer(h, adj_norm)
        return self.head(h)


class MLPBaseline(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.ReLU(), nn.Linear(hidden, 2))

    def forward(self, x, adj_norm=None):
        return self.net(x)

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def train_and_eval(model_cls, log_dir=None, n_train_graphs=200, n_test_graphs=40, epochs=6):
    torch.manual_seed(0)
    model = model_cls().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    writer = SummaryWriter(log_dir=log_dir) if log_dir else None
    rng_train = np.random.RandomState(1)

    step = 0
    for epoch in range(epochs):
        total_loss = 0.0
        for _ in range(n_train_graphs):
            adj, feat, lab = make_graph(rng_train)
            adj_norm = torch.tensor(normalize_adj(adj), device=device)
            x = torch.tensor(feat, device=device).unsqueeze(1)
            y = torch.tensor(lab, dtype=torch.long, device=device)

            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x, adj_norm), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
            if writer:
                writer.add_scalar("loss/train", loss.item(), step)
            step += 1
        print(f"  epoch {epoch}  avg_loss {total_loss / n_train_graphs:.4f}")

    if writer:
        writer.close()

    # Evaluate on freshly-sampled graphs (disjoint seed) - this tests whether the model
    # learned a generalizable rule, not memorized one specific graph.
    rng_test = np.random.RandomState(999)
    accs = []
    with torch.no_grad():
        for _ in range(n_test_graphs):
            adj, feat, lab = make_graph(rng_test)
            adj_norm = torch.tensor(normalize_adj(adj), device=device)
            x = torch.tensor(feat, device=device).unsqueeze(1)
            y = torch.tensor(lab, dtype=torch.long, device=device)
            pred = model(x, adj_norm).argmax(dim=-1)
            accs.append((pred == y).float().mean().item())
    print(f"  held-out accuracy (new graphs): {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
    return np.mean(accs)


print("=== MLP baseline (node features only, no graph) ===")
mlp_acc = train_and_eval(MLPBaseline)
print("=== GCN (message passing over graph structure) ===")
gcn_acc = train_and_eval(GCN, log_dir="/work/runs/gnn")

print()
print(f"MLP baseline (no graph): {mlp_acc:.3f}")
print(f"GCN (graph structure):   {gcn_acc:.3f}")

## Next steps

- Increase `NOISE_STD` until the MLP baseline drops to chance (0.5) - does the GCN still
  recover the communities? There's a limit: at some noise level even structure-aware
  denoising can't help.
- Widen the gap between `P_IN` and `P_OUT` (stronger community structure) or narrow it (weaker)
  and watch the GCN's advantage grow or shrink accordingly.
- Try a real graph library (`torch_geometric` or `dgl`) for larger graphs and more GNN
  variants (GAT, GraphSAGE) - the `GCNLayer` here is the same core idea, just hand-rolled.